In [4]:
import torch
import torch.nn as nn

from ase import units
from ase.atoms import Atoms
from ase.build import molecule
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
from ase.calculators.dftd3 import DFTD3

from cc2cc.utils import gen_mole


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, device="cpu", damping="zero", **kwargs):
        super().__init__()

        # device="cuda:0" for fast GPU computation.
        self.calc = TorchDFTD3Calculator(
            device=device,
            dtype=torch.float64,
            xc="b3-lyp",
            damping=damping,
            bidirectional=False,
        )

        # if damping == "zero":
        #     self.params_vector = torch.nn.Parameter(
        #         torch.tensor(
        #             [
        #                 kwargs.get("s6", 1.0),
        #                 kwargs.get("rs6", 1.261),
        #                 kwargs.get("s18", 1.703),
        #                 kwargs.get("rs18", 1.0),
        #                 kwargs.get("alp", 14.0),
        #             ],
        #             dtype=torch.float64,
        #             device=torch.device(device),
        #         )
        #     )
        #     self.params = {
        #         "s6": kwargs.get("s6", 1.0),
        #         # "s6": self.params_vector[0],
        #         "rs6": self.params_vector[1],
        #         "s18": self.params_vector[2],
        #         "rs18": kwargs.get("rs18", 1.0),
        #         "alp": kwargs.get("alp", 14.0),
        #         # "rs18": self.params_vector[3],
        #         # "alp": self.params_vector[4],
        #     }
        # elif damping == "bj":
        #     self.params_vector = torch.nn.Parameter(
        #         torch.tensor(
        #             [
        #                 kwargs.get("s6", 1.0),
        #                 kwargs.get("rs6", 0.3981),
        #                 kwargs.get("s18", 1.9889),
        #                 kwargs.get("rs18", 4.4211),
        #                 kwargs.get("alp", 14.0),
        #             ],
        #             dtype=torch.float64,
        #             device=torch.device(device),
        #         )
        #     )
        #     self.params = {
        #         "s6": kwargs.get("s6", 1.0),
        #         "rs6": self.params_vector[1],
        #         "s18": self.params_vector[2],
        #         "rs18": self.params_vector[3],
        #         "alp": kwargs.get("alp", 14.0),
        #     }
        # self.calc.dftd_module.params = self.params
        self.damping = damping

    def forward(self, atoms_list):
        self.calc.reset()
        # Calculator.calculate(self, atoms, properties, system_changes)
        input_dicts_list = [self.calc._preprocess_atoms(atoms) for atoms in atoms_list]
        # --- Make batch ---
        n_nodes_list = [d["Z"].shape[0] for d in input_dicts_list]
        shift_index_array = torch.cumsum(torch.tensor([0] + n_nodes_list), dim=0)
        cell_batch = torch.stack(
            [
                (
                    torch.eye(3, device=self.calc.device, dtype=self.calc.dtype)
                    if d["cell"] is None
                    else d["cell"]
                )
                for d in input_dicts_list
            ]
        )

        batch_dicts = dict(
            Z=torch.cat([d["Z"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            pos=torch.cat([d["pos"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            cell=cell_batch,  # (bs, 3, 3)
            pbc=torch.stack([d["pbc"] for d in input_dicts_list]),  # (bs, 3)
            shift_pos=torch.cat(
                [d["shift_pos"] for d in input_dicts_list], dim=0
            ),  # (n_nodes,)
        )
        batch_dicts["edge_index"] = torch.cat(
            [
                d["edge_index"] + shift_index_array[i]
                for i, d in enumerate(input_dicts_list)
            ],
            dim=1,
        )
        batch_dicts["batch"] = torch.cat(
            [
                torch.full((n_nodes,), i, dtype=torch.long, device=self.calc.device)
                for i, n_nodes in enumerate(n_nodes_list)
            ],
            dim=0,
        )
        batch_dicts["batch_edge"] = torch.cat(
            [
                torch.full(
                    (d["edge_index"].shape[1],),
                    i,
                    dtype=torch.long,
                    device=self.calc.device,
                )
                for i, d in enumerate(input_dicts_list)
            ],
            dim=0,
        )

        batch_dicts["pos"].requires_grad_(True)
        E_disp = self.calc.dftd_module.calc_energy_batch(
            **batch_dicts, damping=self.damping
        )

        return E_disp * units.mol / units.kcal


data = pd.read_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-713074_gmtkn-cc-pVDZ.csv"
)
name_mol_list = (data["name"].str.split("_cc-pVDZ").str[0]).to_numpy()
delta_d3zero_list = data["delta_d3zero"].to_numpy() * 627.5094733748099

atoms_list = []
name_list = []
d3zero_list = []
energy_list_direct = []
energy_list_ase_d3 = []
for name_mol in name_mol_list:
    if not name_mol.startswith("BSR36"):
        continue
    mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
    atoms = Atoms(symbols=mol.elements, positions=mol.atom_coords() * units.Bohr)
    atoms_list.append(atoms)
    name_list.append(name_mol)
    d3zero_list.append(delta_d3zero_list[name_mol_list == name_mol][0])

    calc1 = TorchDFTD3Calculator(atoms=atoms, device="cpu", damping="zero", xc="b3-lyp")
    calc1.reset()
    energy_list_direct.append(atoms.get_potential_energy() * units.mol / units.kcal)

    # mf = mol.KS(xc="b3lyp-d3zero")
    # mf.kernel()
    # print(f"{name_mol} energy: {mf.e_tot} kcal/mol")

model = Model(device="cpu", damping="zero")
energy = model(atoms_list)

print(name_list)
print(np.array(energy_list_direct) - d3zero_list)
print(energy.detach().numpy() - d3zero_list)

# target = torch.tensor(
#     [-1, -1, -1],
#     dtype=torch.float64,
#     device=torch.device("cpu"),
# )
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=0.01,
#     weight_decay=1e-5,
# )
# loss_function = torch.nn.MSELoss(reduction="mean")
# torch.set_printoptions(precision=10)

# for epoch in range(10000):
#     optimizer.zero_grad()
#     energy = model(atoms_list)
#     loss = loss_function(energy, target)
#     loss_record = torch.sum(torch.abs(energy - target))
#     # clip the loss to avoid exploding gradients
#     torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#     loss.backward()
#     optimizer.step()

#     if epoch % 100 == 0:
#         print(f"loss: {loss_record.item()}")

# print(f"energy {energy} kcal/mol, params_vector {model.params}")

['BSR36-ch4', 'BSR36-c2h6', 'BSR36-r1', 'BSR36-r2', 'BSR36-r3', 'BSR36-h1', 'BSR36-h2', 'BSR36-h3', 'BSR36-h4', 'BSR36-h5', 'BSR36-r4', 'BSR36-r5', 'BSR36-r6', 'BSR36-r7', 'BSR36-r8', 'BSR36-r9', 'BSR36-h10', 'BSR36-h11', 'BSR36-h12', 'BSR36-h13', 'BSR36-h14', 'BSR36-h6', 'BSR36-h7', 'BSR36-h8', 'BSR36-h9', 'BSR36-c1', 'BSR36-h15', 'BSR36-r10', 'BSR36-r11', 'BSR36-r12', 'BSR36-r13', 'BSR36-r14', 'BSR36-c2', 'BSR36-c3', 'BSR36-c5', 'BSR36-r15', 'BSR36-r16', 'BSR36-c4']
[-8.94519714e-08 -2.35879189e-05 -7.98464176e-05 -1.65366761e-04
  2.21580379e-04 -2.64092274e-04 -1.25048337e-04 -8.48293768e-05
 -1.20913038e-04 -1.09187173e-04 -6.33142299e-04 -7.18612336e-05
 -1.63419109e-04 -2.88449688e-04  8.37309225e-05 -3.12953822e-04
 -2.22546291e-04  1.47210480e-04  1.00044651e-04 -1.17170284e-04
  6.66486959e-05 -6.40598502e-05  1.32104197e-04 -1.14070707e-04
  8.36692374e-05  3.65911284e-04  2.18322280e-04  1.88400672e-04
 -8.23855999e-05 -1.35087882e-04 -2.45774663e-04  9.64265195e-06
 -7.862

# energy -7.968839186200644 kcal/mol
